### Image Data Preprocessing

In [1]:
import torch
from PIL import Image
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

class GTZANImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # get all kind of files
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # go through all of the pic_dir and labels
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            if os.path.isdir(cls_dir):
                for img_name in os.listdir(cls_dir):
                    self.image_paths.append(os.path.join(cls_dir, img_name))
                    self.labels.append(self.class_to_idx[cls_name])
            else:
                print("Image Path error.")

    def __len__(self):
        return len(self.image_paths)
    
    # remove the white borders...
    def _remove_borders(self, image):
        img_np = np.array(image)
        # find legal area
        non_white = np.where(img_np < 240)
        # judge the border
        y_min, y_max = non_white[0].min(), non_white[0].max()
        x_min, x_max = non_white[1].min(), non_white[1].max()

        img_cropped = img_np[y_min:y_max+1, x_min:x_max+1, :]
        
        return Image.fromarray(img_cropped)
    
    def __getitem__(self,idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        image = self._remove_borders(image)
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)

        return image, label


# data pre-processing
transform = transforms.Compose([
    transforms.Resize((180,180)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# input dataset
dataset_path = './data/images_original'
full_dataset = GTZANImageDataset(dataset_path,transform)

# divide dataset 
total_size = len(full_dataset)
train_size = int(0.7*total_size)
val_size = int(0.2*total_size)
test_size = total_size-train_size-val_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size], generator=generator)

# create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


### Music data processing

In [ ]:
import os
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset
import torch.nn as nn


### Net1 Pure MLP

In [9]:
import torch.nn as nn
import torch.nn.functional as F

class Net1(nn.Module):
    def __init__(self):
        super(Net1, self).__init__()
        self.input_size = 3*180*180
        self.hidden_size1 =1024
        self.hidden_size2 = 512
        self.num_classes = 10
        self.flatten = nn.Flatten()
        # add 2 hidden layers
        self.fc1 = nn.Linear(self.input_size, self.hidden_size1)
        self.fc2 = nn.Linear(self.hidden_size1, self.hidden_size2)      
        self.fc3 = nn.Linear(self.hidden_size2, self.num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x



### Net2 Normal CNN

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class Net2(nn.Module):
    def __init__(self, num_classes=10):
        super(Net2, self).__init__()
        # part1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # part2
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 180/2/2
        self.flatten_dim = 64*45*45

        # fully connecting layer
        self.fc1 = nn.Linear(in_features = self.flatten_dim, out_features = 256)
        self.fc2 = nn.Linear(in_features = 256, out_features=num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = self.pool1(x)

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))

        x = self.pool2(x)

        # flatten to 1 dimension
        x = x.view(-1, self.flatten_dim)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### Net3 Batch Normalization

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class Net3(nn.Module):
    def __init__(self, num_classes=10):
        super(Net3, self).__init__()
        # part1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # part2
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten_dim = 64*45*45

        # fully connecting layer
        self.fc1 = nn.Linear(in_features = self.flatten_dim, out_features = 256)
        self.bn5 = nn.BatchNorm2d(256)
        self.fc2 = nn.Linear(in_features = 256, out_features=num_classes)


    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = self.pool1(x)

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))

        x = self.pool2(x)

        # flatten to 1 dimension
        x = x.view(-1, self.flatten_dim)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### Net5 LSTM

In [5]:
import torch.nn as nn

class Net5(nn.Module):
    def __init__(self, input_size=40, hidden_size=128, num_layers=2, num_classes=10):
        super(Net5, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = False

        # batch, seq, feature
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=self.bidirectional)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:,-1,:]
        out = self.fc(out)
        return out
    

### training Engine

In [6]:
import torch
import copy
import time

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device='cpu', patience=None):
    # patience means Early Stopping
    
    model = model.to(device)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    best_acc = 0.0
    epochs_no_improve = 0 # number of rounds with no improvement in val-set loss

    # mark the data for drawing ACC curve
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    start_time = time.time()

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # go through dataset
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # forward and control grad calculation
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs) # batch_size, num_class
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    #backward
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            # recode data
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                print(f'Val Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

                # early Stopping
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

        if patience is not None and epochs_no_improve >= patience:
            print(f'Early stopping triggered after {epoch + 1} epochs.')
            break

    time_elapsed = time.time() - start_time
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # recall best model
    model.load_state_dict(best_model_wts)
    return model, history


### Draw pic

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


def plot_acc_analysis(results_img, save_dir='./', figsize=(16, 8), dpi=300):
    # divide 50 & 100
    exp_50 = [k for k in results_img.keys() if '_50' in k]
    exp_100 = [k for k in results_img.keys() if '_100' in k]
    
    # get best acc
    best_val_acc = {}
    for exp_name, history in results_img.items():
        best_val_acc[exp_name] = max(history['val_acc'])
    
    # set style
    colors = {'Net1': '#1f77b4', 'Net2': '#ff7f0e', 'Net3': '#2ca02c', 'Net4': '#d62728'}
    linestyles = {'train': '-', 'val': '--'}
    

    # ACC pic
    plt.rcParams['font.size'] = 10
    plt.rcParams['figure.figsize'] = figsize
    
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=False, sharey=True)
    fig.suptitle('Net1-Net4 Training/Validation Accuracy Comparison', fontsize=14, fontweight='bold')
    
    # 50 epochs
    ax1.set_title('50 Epochs', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=11)
    ax1.grid(alpha=0.3)
    for exp_name in exp_50:
        net_name = exp_name.split('_')[0]
        history = results_img[exp_name]
        epochs = range(1, len(history['train_acc']) + 1)
        
        ax1.plot(epochs, history['train_acc'], color=colors[net_name], linestyle=linestyles['train'],
                 label=f'{net_name} (train)')
        ax1.plot(epochs, history['val_acc'], color=colors[net_name], linestyle=linestyles['val'],
                 label=f'{net_name} (val)')
    ax1.legend(loc='lower right', framealpha=0.9)
    
    # 100 epochs
    ax2.set_title('100 Epochs', fontsize=12)
    ax2.set_xlabel('Epochs', fontsize=11)
    ax2.set_ylabel('Accuracy', fontsize=11)
    ax2.grid(alpha=0.3)
    for exp_name in exp_100:
        net_name = exp_name.split('_')[0]
        history = results_img[exp_name]
        epochs = range(1, len(history['train_acc']) + 1)
        
        ax2.plot(epochs, history['train_acc'], color=colors[net_name], linestyle=linestyles['train'])
        ax2.plot(epochs, history['val_acc'], color=colors[net_name], linestyle=linestyles['val'])
    ax2.legend(loc='lower right', framealpha=0.9)
    
    plt.tight_layout()
    curve_save_path = f'{save_dir}/acc_curves_subplot.png'
    plt.savefig(curve_save_path, dpi=dpi, bbox_inches='tight')
    print(f'ACC pic has already saved to :{curve_save_path}')
    plt.show()
    

    # bar chart
    plt.rcParams['figure.figsize'] = (12, 6)
    fig, ax = plt.subplots()
    
    exp_names = list(best_val_acc.keys())
    acc_values = [best_val_acc[k] for k in exp_names]
    
    bar_colors = []
    for exp in exp_names:
        net_name = exp.split('_')[0]
        bar_colors.append(colors[net_name] + '80' if '_50' in exp else colors[net_name]) # different color for 50 and 100
    
    bars = ax.bar(exp_names, acc_values, color=bar_colors, edgecolor='black', linewidth=0.5)
    
    # remark value
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    
    ax.set_title('Best Validation Accuracy of All Experiments', fontsize=14, fontweight='bold')
    ax.set_xlabel('Experiment Name', fontsize=11)
    ax.set_ylabel('Best Validation Accuracy', fontsize=11)
    ax.set_ylim(0, max(acc_values) * 1.1)
    ax.grid(axis='y', alpha=0.3)
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='gray', alpha=0.5, label='50 Epochs'),
        Patch(facecolor='gray', alpha=1.0, label='100 Epochs')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    bar_save_path = f'{save_dir}/best_acc_bar.png'
    plt.tight_layout()
    plt.savefig(bar_save_path, dpi=dpi, bbox_inches='tight')
    print(f'bar chart has already saved to :{bar_save_path}')
    plt.show()

### Experiments Execution

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()

# param for net1-4
experiments_img = {
    'Net1_50':  {'model': Net1(), 'optim': optim.Adam, 'lr': 0.001, 'epochs': 50},
    'Net2_50':  {'model': Net2(), 'optim': optim.Adam, 'lr': 0.001, 'epochs': 50},
    'Net3_50':  {'model': Net3(), 'optim': optim.Adam, 'lr': 0.001, 'epochs': 50},
    'Net4_50':  {'model': Net3(), 'optim': optim.RMSprop, 'lr': 0.001, 'epochs': 50},
    'Net1_100':  {'model': Net1(), 'optim': optim.Adam, 'lr': 0.001, 'epochs': 100},
    'Net2_100':  {'model': Net2(), 'optim': optim.Adam, 'lr': 0.001, 'epochs': 100},
    'Net3_100':  {'model': Net3(), 'optim': optim.Adam, 'lr': 0.001, 'epochs': 100},
    'Net4_100': {'model': Net3(), 'optim': optim.RMSprop, 'lr': 0.001, 'epochs': 100}
}

results_img = {}

for exp_name, config in experiments_img.items():
    print(f"\n--- Running Experiment: {exp_name} ---")
    model = config['model']
    optimizer = config['optim'](model.parameters(), lr=config['lr'])
    
    best_model, history = train_model(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        criterion=criterion, 
        optimizer=optimizer, 
        num_epochs=config['epochs'], 
        device=device,
        patience=None
    )
    results_img[exp_name] = history

    plot_acc_analysis(results_img=results_img, save_dir='./experiment_results', figsize=(16, 8), dpi=300)